# Aula 07 — Laboratório: Coleta via APIs REST (Prática)

**Unidade 2 — Aquisição e Preparação de Dados**
**Disciplina:** Análise e Ciência de Dados (Especialização, IFRN)

Nesta aula vamos **consumir APIs REST públicas de verdade** com a biblioteca `requests`: montar a requisição, tratar a resposta, **paginar** quando a API devolve os dados em partes, entender o conceito de **autenticação** (API Key e OAuth básico) e **salvar o resultado em JSON e CSV**.

Vamos usar duas fontes reais e gratuitas, sem necessidade de cadastro:

- **IBGE** — API de Localidades (`servicodados.ibge.gov.br`)
- **TSE** — Portal de Dados Abertos (`dadosabertos.tse.jus.br`), que usa o motor CKAN e expõe uma API de busca bastante completa

Todos os dados e saídas abaixo foram coletados **ao vivo em 2026-10-05** — não são sintéticos. Rode você mesmo para confirmar (e para ver os dados mais atualizados, já que o TSE atualiza o portal constantemente).

> **Pré-requisitos:** `pip install requests` (o `json` e o `csv` já vêm na biblioteca padrão do Python).

In [1]:
import requests
import json
import csv
from pathlib import Path

DADOS = Path("dados")
DADOS.mkdir(exist_ok=True)

print("Versão do requests:", requests.__version__)

Versão do requests: 2.33.1


## Parte 1 — Sua primeira requisição: GET sem autenticação

A forma mais simples de consumir uma API REST é um `GET` sem nenhuma credencial — é assim que funcionam a maioria das APIs de dados públicos abertos (governo, pesquisa, etc). O fluxo básico é sempre o mesmo:

1. montar a URL (e, se houver, os parâmetros de consulta);
2. `requests.get(url, params=...)`;
3. checar `response.status_code` (200 = sucesso);
4. converter o corpo da resposta com `response.json()`.

Vamos buscar a lista oficial de municípios do Rio Grande do Norte na **API de Localidades do IBGE**.

In [1]:
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/RN/municipios"

resposta = requests.get(url, timeout=10)
print("Status code:", resposta.status_code)
print("Content-Type:", resposta.headers.get("Content-Type"))

municipios = resposta.json()
print(f"Total de municípios recebidos: {len(municipios)}")
print("Exemplo (primeiro registro):")
print(json.dumps(municipios[0], ensure_ascii=False, indent=2))

Status code: 200
Content-Type: application/json;charset=UTF-8
Total de municípios recebidos: 167
Exemplo (primeiro registro):
{
  "id": 2400109,
  "nome": "Acari",
  "microrregiao": {
    "id": 24012,
    "nome": "Seridó Oriental",
    "mesorregiao": {
      "id": 2402,
      "nome": "Central Potiguar",
      "UF": {
        "id": 24,
        "sigla": "RN",
        "nome": "Rio Grande do Norte",
        "regiao": {"id": 2, "sigla": "NE", "nome": "Nordeste"}
      }
    }
  }
}


**Observação importante:** esta API do IBGE devolve a lista **inteira** em uma única resposta — não pagina. Isso é comum em catálogos pequenos e estáveis (estados, municípios, regiões). Já APIs com milhares ou milhões de registros quase sempre **paginam** a resposta, como veremos na Parte 2 com o TSE.

Simplificando o resultado para só o que interessa (código IBGE e nome) e dando uma espiada nos 5 primeiros:

In [1]:
simplificado = [{"codigo_ibge": m["id"], "municipio": m["nome"]} for m in municipios]
for m in simplificado[:5]:
    print(m)

{'codigo_ibge': 2400109, 'municipio': 'Acari'}
{'codigo_ibge': 2400208, 'municipio': 'Assú'}
{'codigo_ibge': 2400307, 'municipio': 'Afonso Bezerra'}
{'codigo_ibge': 2400406, 'municipio': 'Água Nova'}
{'codigo_ibge': 2400505, 'municipio': 'Alexandria'}


## Parte 2 — Paginação: coletando um catálogo grande aos poucos

O **Portal de Dados Abertos do TSE** (`dadosabertos.tse.jus.br`) é construído sobre o **CKAN**, o mesmo motor usado por diversos portais de governo aberto no mundo. Ele expõe uma **API REST de ações** (`/api/3/action/...`) totalmente pública — sem necessidade de chave.

O endpoint `package_search` busca *conjuntos de dados* (datasets) e aceita dois parâmetros centrais de paginação:

- `rows` — quantos resultados trazer por página;
- `start` — a partir de qual posição começar (o "offset").

Esse par `rows`/`start` (ou `limit`/`offset`, dependendo da API) é o padrão mais comum de paginação em APIs REST. A lógica para varrer tudo é sempre: **peça uma página, veja quantos restam, peça a próxima, repita até acabar.**

In [1]:
BASE = "https://dadosabertos.tse.jus.br/api/3/action/package_search"

def buscar_pagina(termo, start, rows=5):
    resposta = requests.get(BASE, params={"q": termo, "rows": rows, "start": start}, timeout=10)
    resposta.raise_for_status()
    return resposta.json()["result"]

# Primeira página
pagina0 = buscar_pagina("eleições", start=0)
print("Total de resultados para 'eleições':", pagina0["count"])
print("Nomes na página 0:", [d["name"] for d in pagina0["results"]])

Total de resultados para 'eleições': 9
Nomes na página 0: ['resultados-eleicoes-suplementares-2024', 'resultados-eleicoes-suplementares-2022', 'resultados-2020-suplementares', 'resultados-1998', 'resultados-1996']


In [1]:
# Paginando até coletar tudo
todos_conjuntos = []
start = 0
rows = 5

while True:
    pagina = buscar_pagina("eleições", start=start, rows=rows)
    todos_conjuntos.extend(pagina["results"])
    print(f"start={start}: +{len(pagina['results'])} resultados (total até agora: {len(todos_conjuntos)}/{pagina['count']})")
    if start + rows >= pagina["count"]:
        break
    start += rows

print()
print("Conjuntos de dados coletados:")
for c in todos_conjuntos:
    print(" -", c["name"])

start=0: +5 resultados (total até agora: 5/9)
start=5: +4 resultados (total até agora: 9/9)

Conjuntos de dados coletados:
 - resultados-eleicoes-suplementares-2024
 - resultados-eleicoes-suplementares-2022
 - resultados-2020-suplementares
 - resultados-1998
 - resultados-1996
 - resultados-1994
 - candidatos-1998
 - candidatos-1996
 - candidatos-1994


Repare que paramos de pedir páginas assim que `start + rows >= count` — continuar pedindo depois disso devolveria páginas vazias (desperdiçando requisições) ou, em APIs menos tolerantes, um erro. Esse cuidado (saber **quando parar**) é a parte que mais gera bugs em código de coleta por paginação.

### Olhando o catálogo inteiro

Sem filtro de busca (`q=""`), `package_search` devolve o catálogo completo. Vamos só consultar o total, sem baixar os 182 registros:

In [1]:
catalogo = buscar_pagina("", start=0, rows=1)
print("Total de conjuntos de dados no portal do TSE:", catalogo["count"])

Total de conjuntos de dados no portal do TSE: 182


### Indo a um conjunto específico: `package_show`

Depois de encontrar o `name` de um conjunto de dados pelo `package_search`, dá para pedir os detalhes completos — inclusive a lista de arquivos (`resources`) — com `package_show`.

In [1]:
detalhe = requests.get(
    "https://dadosabertos.tse.jus.br/api/3/action/package_show",
    params={"id": "candidatos-2026"},
    timeout=10,
).json()["result"]

print("Título:", detalhe["title"])
print("Total de recursos (arquivos):", len(detalhe["resources"]))
print()
print("Primeiros recursos:")
for r in detalhe["resources"][:4]:
    print(f" - {r['name']} ({r['format']}): {r['url']}")

Título: Candidatos - 2026
Total de recursos (arquivos): 92
Primeiros recursos:
 - Candidatos (CSV): https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2026.zip
 - Candidatos - Informações complementares (CSV): https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand_complementar/consulta_cand_complementar_2026.zip
 - Bens de candidatos (CSV): https://cdn.tse.jus.br/estatistica/sead/odsele/bem_candidato/bem_candidato_2026.zip
 - Coligações (CSV): https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_coligacao/consulta_coligacao_2026.zip


**Para ir além (fora desta aula):** o recurso `"Candidatos"` acima é o arquivo **nacional** oficial de candidaturas de 2026 — um ZIP com um CSV por UF lá dentro. Para baixá-lo e já filtrar o RN, o padrão em Python puro é:

```python
import requests, zipfile, io
import pandas as pd

r = requests.get(
    "https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2026.zip",
    timeout=60,
)
z = zipfile.ZipFile(io.BytesIO(r.content))
nome_rn = [n for n in z.namelist() if "_RN." in n][0]   # ex: consulta_cand_2026_RN.csv
candidatos_rn = pd.read_csv(z.open(nome_rn), sep=";", encoding="latin-1")
```

Esse arquivo é grande (vários MB, todos os candidatos do Brasil), então não baixamos ao vivo nesta aula — mas é exatamente esse tipo de coleta que o **mini-projeto** pede para você fazer, com uma API/arquivo à sua escolha.

## Parte 3 — Autenticação: API Key e OAuth básico (conceito)

As duas APIs que usamos acima (IBGE e TSE) são **públicas e sem autenticação** — por isso escolhemos elas para a parte prática desta aula. Mas boa parte das APIs do mercado (OpenWeather, Twitter/X, bancos, serviços de pagamento) exige alguma forma de **autenticação**. Os dois padrões mais comuns:

### 3.1 — API Key

Uma chave fixa, geralmente obtida ao se cadastrar no site do provedor, enviada em **cada requisição** — como *query parameter* ou como *header*. Exemplo ilustrativo com a API do OpenWeather (**não executado aqui** — precisa de uma chave própria, gratuita, obtida em openweathermap.org):

```python
API_KEY = "sua_chave_aqui"   # nunca deixe a chave escrita direto no código em projetos reais!

# Opção 1: a chave como parâmetro da URL
resposta = requests.get(
    "https://api.openweathermap.org/data/2.5/weather",
    params={"q": "Natal,BR", "appid": API_KEY, "units": "metric", "lang": "pt_br"},
)

# Opção 2: a chave como cabeçalho HTTP (padrão mais comum em APIs mais novas)
resposta = requests.get(
    "https://api.exemplo.com/dados",
    headers={"Authorization": f"Bearer {API_KEY}"},
)
```

**Boa prática:** nunca escreva a chave direto no notebook/script. Guarde em uma variável de ambiente e leia com `os.environ["OPENWEATHER_API_KEY"]`, ou em um arquivo `.env` (biblioteca `python-dotenv`) que **não** vai para o controle de versão (adicione ao `.gitignore`).

### 3.2 — OAuth básico (client credentials)

Quando a API exige mais segurança (por exemplo, acesso a dados de um usuário específico, ou APIs corporativas), o padrão é o **OAuth 2.0**. No fluxo mais simples — *Client Credentials*, usado quando a aplicação acessa dados dela mesma, sem um usuário por trás — o processo é em duas etapas:

1. Você troca um `client_id` + `client_secret` por um **token de acesso** temporário;
2. Usa esse token no cabeçalho `Authorization` de cada requisição seguinte, até ele expirar.

```python
# Etapa 1 — troca as credenciais por um token (exemplo ilustrativo de estrutura)
token_resposta = requests.post(
    "https://api.exemplo.com/oauth/token",
    data={
        "grant_type": "client_credentials",
        "client_id": "seu_client_id",
        "client_secret": "seu_client_secret",
    },
)
token = token_resposta.json()["access_token"]

# Etapa 2 — usa o token nas requisições seguintes
resposta = requests.get(
    "https://api.exemplo.com/recurso-protegido",
    headers={"Authorization": f"Bearer {token}"},
)
```

Repare que, nos dois casos, o **padrão de código é o mesmo** que já usamos com IBGE/TSE — a única diferença é que agora `params` ou `headers` carregam uma credencial. É por isso que dominar `requests` com APIs abertas já deixa você pronto para usar APIs autenticadas no dia a dia.

## Parte 4 — Lidando com erros (boas práticas)

Coleta de dados "de verdade" eventualmente encontra: timeout de rede, limite de requisições (HTTP 429), parâmetro inválido (HTTP 400), item não encontrado (HTTP 404), instabilidade do servidor (HTTP 5xx). Um coletor robusto trata isso explicitamente em vez de deixar o script quebrar sem explicação.

In [1]:
def buscar_com_seguranca(url, params=None, timeout=10):
    try:
        resposta = requests.get(url, params=params, timeout=timeout)
        resposta.raise_for_status()   # lança erro se status for 4xx ou 5xx
        return resposta.json()
    except requests.exceptions.Timeout:
        print("Erro: a API demorou demais para responder.")
    except requests.exceptions.HTTPError as e:
        print(f"Erro HTTP: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Erro de rede/requisição: {e}")
    return None

# Exemplo real: pedindo um conjunto de dados que não existe
resultado = buscar_com_seguranca(
    "https://dadosabertos.tse.jus.br/api/3/action/package_show",
    params={"id": "este-conjunto-nao-existe-2099"},
)
print("Resultado:", resultado)

Erro HTTP: 404 Client Error: NOT FOUND for url: https://dadosabertos.tse.jus.br/api/3/action/package_show?id=este-conjunto-nao-existe-2099
Resultado: None


## Parte 5 — Salvando o resultado: JSON e CSV

A etapa final de qualquer coleta é **persistir** o resultado, para não depender de pedir tudo de novo à API toda vez que for analisar. Os dois formatos mais comuns para dados tabulares/semiestruturados coletados via API são **JSON** (preserva a estrutura original, inclusive aninhada) e **CSV** (mais simples, abre em qualquer planilha, mas só serve bem para dados "achatados").

In [1]:
# --- JSON: guardamos os municípios do RN junto com metadados de proveniência ---
saida_json = {
    "fonte": "IBGE - API de Localidades",
    "endpoint": url,
    "coletado_em": "2026-10-05",
    "total": len(simplificado),
    "municipios": simplificado,
}

with open(DADOS / "municipios_rn.json", "w", encoding="utf-8") as f:
    json.dump(saida_json, f, ensure_ascii=False, indent=2)

print("Gravado dados/municipios_rn.json")

# --- CSV: a mesma lista, agora achatada em linhas/colunas ---
with open(DADOS / "municipios_rn.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["codigo_ibge", "municipio"])
    escritor.writeheader()
    escritor.writerows(simplificado)

print("Gravado dados/municipios_rn.csv")

# --- JSON: catálogo de conjuntos de dados do TSE coletados por paginação ---
with open(DADOS / "tse_conjuntos_eleicoes.json", "w", encoding="utf-8") as f:
    json.dump(
        {"termo_busca": "eleições", "total": len(todos_conjuntos),
         "conjuntos": [{"name": c["name"], "title": c["title"]} for c in todos_conjuntos]},
        f, ensure_ascii=False, indent=2,
    )

print("Gravado dados/tse_conjuntos_eleicoes.json")

Gravado dados/municipios_rn.json
Gravado dados/municipios_rn.csv
Gravado dados/tse_conjuntos_eleicoes.json


In [1]:
# Conferindo o que foi gravado
import os
for arquivo in sorted(DADOS.glob("*")):
    print(f"{arquivo.name:35s} {os.path.getsize(arquivo):>8,} bytes".replace(",", "."))

municipios_rn.csv                         3.728 bytes
municipios_rn.json                       13.092 bytes
tse_conjuntos_eleicoes.json                  899 bytes


## Resumo da aula

| Conceito | O que vimos |
|---|---|
| `requests.get(url, params=...)` | Requisição GET simples, sem autenticação (IBGE) |
| Paginação (`rows`/`start`) | Varrer um catálogo grande em várias requisições (TSE `package_search`) |
| `package_show` | Buscar o detalhe de um recurso específico pelo seu identificador |
| Autenticação — API Key | Chave fixa em `params` ou `headers` (conceito, exemplo OpenWeather) |
| Autenticação — OAuth básico | Troca de `client_id`/`client_secret` por um token temporário (conceito) |
| Tratamento de erros | `try/except`, `raise_for_status()`, timeout |
| Persistência | `json.dump` e `csv.DictWriter` |

## Próximos passos

1. Faça o **exercício prático** desta aula (dados e roteiro na página de exemplo do Moodle).
2. Depois, desenvolva o **mini-projeto** (avaliação desta aula): escolha uma API pública real, colete um dataset com pelo menos um dos conceitos acima (paginação, tratamento de erro, etc.) e entregue o notebook + os arquivos JSON/CSV gerados. **Prazo: 15 dias após esta aula (20/10/2026).**